In [1]:
from typing_extensions import TypedDict, Literal, Annotated
from typing import List
from langgraph.types import Send
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel
from operator import add

llm = init_chat_model("openai:gpt-4o-mini")

In [2]:
class State(TypedDict):
    document: str
    final_summary: str
    summaries: Annotated[list[dict], add]

In [3]:
def summarize_p(args):
    paragraph = args["paragraph"]
    index = args["index"]
    response = llm.invoke(
        f"Write a 3-sentence summary for this paragraph: {paragraph}",
    )
    return {
        "summaries": [
            {
                "summary": response.content,
                "index": index,
            }
        ],
    }


def dispatch_summarizers(state: State):
    chunks = state["document"].split("\n\n")
    return [
        Send("summarize_p", {"paragraph": chunk, "index": index}) for index, chunk in enumerate(chunks)
    ]


def final_summary(state: State):
    # index를 넘기기 때문에 요약이 순차적으로 이뤄지지 않아도 llm이 순서를 이해할 수 있음
    response = llm.invoke(
        f"Using the following summaries, give me a final one {state["summaries"]}"
    )
    return {
        "final_summary": response.content,
    }

In [4]:
graph_builder = StateGraph(State)

graph_builder.add_node("summarize_p", summarize_p)
graph_builder.add_node("final_summary", final_summary)


graph_builder.add_conditional_edges(
    START,
    dispatch_summarizers,
    ["summarize_p"],
)

graph_builder.add_edge("summarize_p", "final_summary")
graph_builder.add_edge("final_summary", END)


graph = graph_builder.compile()

In [5]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()


for chunk in graph.stream(
    {"document": document},
    stream_mode="updates",
):
    print(chunk, "\n")

{'summarize_p': {'summaries': [{'summary': 'Inflation for goods has increased compared to earlier in the year, while disinflation persists in the services sector. Recent measures of inflation have shown volatility, influenced by changes in tariffs. Overall, the current inflation dynamics reflect a disparity between goods and services.', 'index': 11}]}} 

{'summarize_p': {'summaries': [{'summary': "Business investment, particularly in equipment and intangible assets, has increased compared to last year's levels. This upward trend indicates a growing confidence among businesses in expanding their operations. Overall, the rise in investment suggests a positive economic outlook.", 'index': 3}]}} 

{'summarize_p': {'summaries': [{'summary': 'Labor demand is weakening, leading to a slowdown in job creation. The current rate of job growth is insufficient to maintain the unemployment rate. As a result, without an increase in hiring, unemployment may rise.', 'index': 7}]}} 

{'summarize_p': {'s